In [4]:
from path import Path

root = Path(r'/home/flo/repos/SystemSimulation/demos/ControlledPendulum/')
pkg_dir = Path(root / "ControlledPendulum")
pkg_str = str(root / "ControlledPendulum" / "package.mo")
pkg_name = "ControlledPendulum"

model_names = Path(pkg_dir).files("*.mo")
model_names = [m.stem for m in model_names if m.stem != "package" and m.stem.startswith("Demo")]

for model_name in model_names:
    print(f"Found model: {model_name}")

Found model: Demo_Driven
Found model: Demo_UndrivenWithWall
Found model: Demo_UndrivenWallDiscrete
Found model: Demo_DrivenWithWall


In [5]:
from OMPython import ModelicaSystem
import shutil

def create_modelica_system(model_name):
    model = ModelicaSystem(pkg_str, model_name, verbose=True)
    return model

model_name = pkg_name + "." + "Demo_Driven"
demo_model = create_modelica_system(model_name)
demo_model.buildModel()


Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.




In [ ]:
# Get involved components and their variables
states = demo_model.getContinuous()

component_dict = {}

for state in states:
    # Check if the state name contains a dot
    if '.' in state:
        component_name, var_name = state.split('.', 1)
        # Remove ( and ) if present
        var_name = var_name.replace('(', '').replace(')', '')
        if component_name not in component_dict:
            component_dict[component_name] = []
        component_dict[component_name].append(var_name)

# Print the components and their variables
for component, variables in component_dict.items():
    print(f"Component: {component}")
    for var in variables:
        print(f"  Variable: {var}")
    print()

Component: drive
  Variable: I
  Variable: U
  Variable: n
  Variable: torque
  Variable: omega
  Variable: u_control

Component: pendulum
  Variable: omega_state
  Variable: q_state
  Variable: torque

Component: pid
  Variable: D.x
  Variable: I.y
  Variable: D.y
  Variable: I.local_set
  Variable: P.y
  Variable: addErr.y
  Variable: lim.simplifiedExpr
  Variable: sumPID.y
  Variable: u
  Variable: D.u
  Variable: I.u
  Variable: P.u
  Variable: addErr.u1
  Variable: addErr.u2
  Variable: gainK.u
  Variable: gainK.y
  Variable: lim.u
  Variable: lim.y
  Variable: ref
  Variable: sumPID.u1
  Variable: sumPID.u2
  Variable: sumPID.u3
  Variable: y

Component: der(drive
  Variable: I

Component: der(pendulum
  Variable: omega_state
  Variable: q_state

Component: der(pid
  Variable: D.x
  Variable: I.y

Component: reference
  Variable: q_ref

Component: sensor_ref
  Variable: U_a
  Variable: U_q
  Variable: alpha
  Variable: q

Component: sensor_state
  Variable: U_a
  Variable: U_q
  

In [12]:
# Set simulation parameters
simulation_options = [
    "startTime=0.0",
    "stopTime=5.0",        
    "stepSize=0.01",
    "tolerance=1e-6"
]

demo_model.setSimulationOptions(simulation_options)
current_options = demo_model.getSimulationOptions()
print(f"Simulation options:")
for option in current_options:
    print(f"  {option}: {current_options[option]}")

Simulation options:
  startTime: 0.0
  stopTime: 5.0
  stepSize: 0.01
  tolerance: 1e-6
  solver: dassl
  outputFormat: mat


In [13]:
# Run simulation
print("Running simulation...")
demo_model.simulate()
print("Simulation completed successfully")

Running simulation...
LOG_SUCCESS       | info    | The initialization finished successfully without homotopy method.
LOG_SUCCESS       | info    | The simulation finished successfully.
Simulation completed successfully


In [14]:
# Extract results
results = demo_model.getSolutions()

In [15]:
results

('der(pendulum.omega_state)',
 'der(pendulum.q_state)',
 'drive.U_M',
 'drive.omega',
 'drive.torque',
 'drive.u_control',
 'pendulum.L',
 'pendulum.m',
 'pendulum.omega0',
 'pendulum.omega_state',
 'pendulum.q0',
 'pendulum.q_state',
 'pendulum.torque',
 'pid.Nd',
 'pid.Td',
 'pid.Ti',
 'pid.k',
 'pid.ref',
 'pid.u',
 'pid.uMax',
 'pid.uMin',
 'pid.y',
 'reference.amplitude',
 'reference.frequency',
 'reference.mean',
 'reference.q_ref',
 'sensor_ref.U_q',
 'sensor_ref.nBits',
 'sensor_ref.q',
 'sensor_ref.q_max',
 'sensor_ref.q_min',
 'sensor_state.U_q',
 'sensor_state.nBits',
 'sensor_state.q',
 'sensor_state.q_max',
 'sensor_state.q_min',
 'time')

In [16]:
time_values = demo_model.getSolutions('time')
ref_values = demo_model.getSolutions('reference.q_ref')
state_values = demo_model.getSolutions('pendulum.q_state')

In [17]:
time_values = time_values.flatten()
ref_values = ref_values.flatten()
state_values = state_values.flatten()

In [19]:
# Create a simple plotly figure for the q_ref and q_state over time
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=time_values, y=ref_values, mode='lines',
                            name='Reference Angle (q_ref)', line=dict(color='white', dash='dash')))
fig.add_trace(go.Scatter(x=time_values, y=state_values, mode='lines',
                            name='State Angle (q_state)', line=dict(color='red')))
fig.update_layout(title='Pendulum Angle Tracking',
                  xaxis_title='Time (s)',
                  yaxis_title='Angle (degrees)',
                  legend_title='Legend',
                  template='plotly_dark')
fig.show()